# Part III · Lesson 5 — Choose a detail or the bigger picture

## Import the document and model helpers

In [ ]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.documents import Document
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

## Connect the two kinds of model

In [ ]:
load_dotenv()
# Groq generates answers; OpenAI creates search embeddings.
model = init_chat_model(
    "openai/gpt-oss-120b", model_provider="groq", reasoning_effort="low",
)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

## Read the structured catalog

In [ ]:
import json
from pathlib import Path

catalog_path = Path("../doc/products2/Hierarchical_Product_Catalog.json")
catalog = json.loads(catalog_path.read_text(encoding="utf-8"))
products = catalog["products"]
print("Products:", len(products))

## Build one detailed document per product

In [ ]:
product_documents = []
for product in products:
    text = "\n".join([
        f"Product: {product['product_name']} ({product['product_id']})",
        f"Subcategory: {product['subcategory']}", product["description"],
        "Specifications: " + json.dumps(product["specifications"]),
        "Setup: " + " ".join(product["quick_start"]),
        "Usage: " + " ".join(product["how_to_use"]),
        "Errors: " + json.dumps(product["error_codes"]),
        "Variants: " + json.dumps(product["variants"]),
    ])
    product_documents.append(Document(page_content=text, metadata={
        "source": catalog_path.name, "level": "product",
        "product_id": product["product_id"], "subcategory": product["subcategory"],
    }))

## Give each variant its own searchable document

In [ ]:
variant_documents = []
for product in products:
    for variant in product["variants"]:
        text = (
            f"Parent: {product['product_name']} ({product['product_id']})\n"
            f"Variant: {variant['variant_name']} ({variant['variant_id']})\n"
            f"Price: {variant['price_usd']}\n{variant['description']}"
        )
        variant_documents.append(Document(page_content=text, metadata={
            "source": catalog_path.name, "level": "variant",
            "variant_id": variant["variant_id"], "parent_product_id": product["product_id"],
            "subcategory": product["subcategory"],
        }))

## Group products by their place in the hierarchy

In [ ]:
groups = {}
for product in products:
    key = (product["category"], product["subcategory"])
    groups.setdefault(key, []).append(product)

## Create a complete summary for each group

In [ ]:
summary_documents = []
for (category, subcategory), group in groups.items():
    lines = [f"Category: {category}", f"Subcategory: {subcategory}", f"Products: {len(group)}"]
    for product in group:
        lines.append(
            f"{product['product_name']} ({product['product_id']}): {product['description']} "
            f"Starting price: {product['starting_price_usd']}. Variants: {len(product['variants'])}."
        )
    summary_documents.append(Document(page_content="\n".join(lines), metadata={
        "source": catalog_path.name, "level": "summary", "category": category,
        "subcategory": subcategory, "product_count": len(group),
    }))

## Prepare stable IDs for the detail documents

In [ ]:
detail_documents = product_documents + variant_documents
detail_ids = [
    f"product:{doc.metadata['product_id']}" for doc in product_documents
] + [
    f"variant:{doc.metadata['variant_id']}" for doc in variant_documents
]

## Save the detail collection in Chroma

In [ ]:
detail_store = Chroma.from_documents(
    documents=detail_documents,
    embedding=embeddings,
    ids=detail_ids,
    collection_name="detail_documents",
    persist_directory="./chroma_db",
)

## Save the summary collection in Chroma

In [ ]:
summary_store = Chroma.from_documents(
    documents=summary_documents,
    embedding=embeddings,
    ids=[f"{doc.metadata['category']}:{doc.metadata['subcategory']}" for doc in summary_documents],
    collection_name="summary_documents",
    persist_directory="./chroma_db",
)

## Choose how much text each route retrieves

In [ ]:
detail_retriever = detail_store.as_retriever(search_kwargs={"k": 3})
# Include every summary for complete coverage of this small catalog.
summary_retriever = summary_store.as_retriever(search_kwargs={"k": len(summary_documents)})

## Define the allowed route and working state

In [ ]:
from typing import Literal
from typing_extensions import TypedDict
from pydantic import BaseModel
from langgraph.graph import StateGraph, START, END

class RouteChoice(BaseModel):
    route: Literal["DETAIL", "SUMMARY"]

class CatalogState(TypedDict):
    question: str
    route: str
    documents: list[Document]
    answer: str

## Ask the model to classify the question

In [ ]:
route_model = model.with_structured_output(RouteChoice)

def classify_question(state: CatalogState) -> dict:
    prompt = (
        "Choose DETAIL for a specific product, variant, specification, error, or setup question. "
        "Choose SUMMARY for a comparison, overview, or list across products or categories.\n"
        "Question: " + state["question"]
    )
    choice = route_model.invoke(prompt)
    return {"route": choice.route}

def choose_route(state: CatalogState) -> str:
    return state["route"]

## Retrieve at the selected level

In [ ]:
def retrieve_detail(state: CatalogState) -> dict:
    return {"documents": detail_retriever.invoke(state["question"])}

def retrieve_summary(state: CatalogState) -> dict:
    return {"documents": summary_retriever.invoke(state["question"])}

## Label context with its level and identity

In [ ]:
def catalog_context(documents: list[Document]) -> str:
    passages = []
    for document in documents:
        meta = document.metadata
        label = meta.get("variant_id", meta.get("product_id", meta["subcategory"]))
        passages.append(f"[{label}; {meta['level']}; {meta['source']}]\n{document.page_content}")
    return "\n\n".join(passages)

## Generate an answer using that context

In [ ]:
def generate_catalog_answer(state: CatalogState) -> dict:
    instruction = (
        "Use only the context. If it lacks the needed detail, say you could not find enough information. "
        "Cite the bracketed source labels.\nContext:\n" + catalog_context(state["documents"])
    )
    answer = model.invoke([SystemMessage(instruction), HumanMessage(state["question"])]).text
    return {"answer": answer}

## Register the classification, retrieval, and answer nodes

In [ ]:
builder = StateGraph(CatalogState)
builder.add_node("route", classify_question)
builder.add_node("detail", retrieve_detail)
builder.add_node("summary", retrieve_summary)
builder.add_node("generate", generate_catalog_answer)

## Connect the two retrieval routes

In [ ]:
builder.add_edge(START, "route")
builder.add_conditional_edges(
    "route", choose_route, {"DETAIL": "detail", "SUMMARY": "summary"},
)
builder.add_edge("detail", "generate")
builder.add_edge("summary", "generate")
builder.add_edge("generate", END)
graph = builder.compile()

## Run a question with fresh state

In [ ]:
def ask_hierarchical_rag(question: str) -> CatalogState:
    return graph.invoke({
        "question": question, "route": "", "documents": [], "answer": "",
    })

## Ask for a specific variant feature

In [ ]:
detail_result = ask_hierarchical_rag("Which AeroTherm X1 variant adds humidity control?")
print("Route:", detail_result["route"])
print("Answer:", detail_result["answer"])
for document in detail_result["documents"]:
    print(document.metadata)

## Ask for a view across products

In [ ]:
summary_result = ask_hierarchical_rag("Compare all Climate Control products.")
print("Route:", summary_result["route"])
print("Summary documents:", len(summary_result["documents"]))
print("Answer:", summary_result["answer"])